In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/trash2

In [ ]:
import os

path = "/content/drive/MyDrive/trash2"

In [ ]:
!pip install ultralytics

#TACO

In [ ]:
#데이터셋 다운로드

!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="l89cTNFbkPehbFQyMRXy")
project = rf.workspace("seungwoo-ydi9o").project("taco-dataset-brnom-ltugr")
version = project.version(1)
dataset = version.download("yolov8")


In [ ]:
# 클래스 59개 -> 8개

import os

# 원본 클래스명 리스트 (TACO 데이터셋 등 실제 원본 클래스명 소문자화)
original_classes = [
    "aerosol", "aluminium blister pack", "aluminium foil", "battery", "broken glass",
    "carded blister pack", "cigarette", "clear plastic bottle", "corrugated carton",
    "crisp packet", "disposable food container", "disposable plastic cup", "drink can",
    "drink carton", "egg carton", "foam cup", "foam food container", "food can",
    "food waste", "garbage bag", "glass bottle", "glass cup", "glass jar", "magazine paper",
    "meal carton", "metal bottle cap", "metal lid", "normal paper", "other carton",
    "other plastic", "other plastic bottle", "other plastic container", "other plastic cup",
    "other plastic wrapper", "paper bag", "paper cup", "paper straw", "pizza box",
    "plastic bottle cap", "plastic film", "plastic glooves", "plastic lid", "plastic straw",
    "plastic utensils", "polypropylene bag", "pop tab", "rope & strings", "scrap metal",
    "shoe", "single-use carrier bag", "six pack rings", "spread tub", "squeezable tube",
    "styrofoam piece", "tissues", "toilet tube", "tupperware", "unlabeled litter", "wrapping paper"
]

# 최종 사용할 8개 클래스와 포함된 기존 클래스 이름들 매핑
class_mapping = {
    "vinyl": [  # 비닐류
        "garbage bag", "polypropylene bag", "single-use carrier bag", "plastic film"
    ],
    "styrofoam": [  # 스티로폼류
        "styrofoam piece", "foam cup", "foam food container"
    ],
    "paper": [  # 종이류
        "corrugated carton", "egg carton", "pizza box", "meal carton", "other carton",
        "paper bag", "paper cup", "magazine paper", "normal paper", "wrapping paper",
        "tissues", "toilet tube", "paper straw"
    ],
    "can": [  # 캔류
        "drink can", "food can", "metal bottle cap", "metal lid", "pop tab", "six pack rings",
        "aerosol"
    ],
    "pet_bottle": [  # 페트병
        "clear plastic bottle", "other plastic bottle"
    ],
    "plastic": [  # 플라스틱류
        "other plastic", "other plastic container", "other plastic cup", "other plastic wrapper",
        "plastic bottle cap", "plastic glooves", "plastic lid", "plastic straw",
        "plastic utensils", "spread tub", "squeezable tube", "tupperware"
    ],
    "cigarette": [  # 담배꽁초
        "cigarette"
    ],
    "etc": [  # 기타
        "broken glass", "battery", "rope & strings", "scrap metal", "shoe",
        "unlabeled litter", "glass bottle", "glass cup", "glass jar", "carded blister pack",
        "aluminium blister pack", "aluminium foil", "disposable food container",
        "disposable plastic cup", "drink carton"
    ]
}

# 기존 클래스 인덱스 (0~len(original_classes)-1) → 클래스명 (소문자)
idx_to_class = {i: name for i, name in enumerate(original_classes)}

# 8개 클래스명 → 새로운 인덱스 (0~7)
new_class_idx = {name: i for i, name in enumerate(class_mapping.keys())}

# 기존 클래스명이 8개 클래스 어느 그룹에 속하는지 찾는 함수
def find_new_class(old_class_name):
    old_class_name = old_class_name.lower()
    for new_cls, old_names in class_mapping.items():
        if old_class_name in old_names:
            return new_class_idx[new_cls]
    return None  # 매핑 안 되는 클래스는 None 처리 (라벨 제거)

# 단일 라벨 파일 처리
def process_label_file(txt_path, output_path):
    new_lines = []
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            old_cls_idx = int(parts[0])
            old_cls_name = idx_to_class.get(old_cls_idx)
            if old_cls_name is None:
                continue
            new_cls = find_new_class(old_cls_name)
            if new_cls is None:
                continue
            new_line = f"{new_cls} " + " ".join(parts[1:])
            new_lines.append(new_line)
    if new_lines:
        with open(output_path, 'w') as f_out:
            f_out.write("\n".join(new_lines))

# 폴더 단위 처리
def process_labels_folder(input_label_dir, output_label_dir):
    os.makedirs(output_label_dir, exist_ok=True)
    for fname in os.listdir(input_label_dir):
        if not fname.endswith(".txt"):
            continue
        input_path = os.path.join(input_label_dir, fname)
        output_path = os.path.join(output_label_dir, fname)
        process_label_file(input_path, output_path)
    print(f"라벨 재처리 완료: {output_label_dir}")

folders = ["train", "valid", "test"]  # 실제 폴더명에 맞게 수정하세요

base_dir = os.path.join(path, "TACO-dataset-1")  # 실제 경로 맞춰주세요

for folder in folders:
    input_label_dir = os.path.join(base_dir, folder, "labels")
    output_label_dir = os.path.join(base_dir, folder, "labels_8class")
    process_labels_folder(input_label_dir, output_label_dir)


In [ ]:
from ultralytics import YOLO

# 사전 학습된 YOLOv8s 모델 사용
model = YOLO("yolov8s.pt")

# 훈련
model.train(
    data=os.path.join(path, "TACO-dataset-1/data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    project=os.path.join(path, "trash_yolo"),
    name="model_t",
    exist_ok=True
)


In [ ]:
model = YOLO(os.path.join(path, "trash_yolo/model_t/weights/best.pt"))

results = model(os.path.join(path, "test_img/t3.jpg"))  # 예측
results[0].show()  # 이미지에 박스 그려진 결과 보기


#household_waste

In [ ]:
# (json -> txt) & (train/val/test 분할)

import os
import json
import random
import shutil
from glob import glob

# ===== 경로 설정 =====
json_dir = os.path.join(path, "household_waste/labels")
image_dir = os.path.join(path, "household_waste/images")

# 변환된 YOLO 라벨 저장 경로
yolo_label_dir = os.path.join(path, "household_waste/yolo_labels")
os.makedirs(yolo_label_dir, exist_ok=True)

# 최종 dataset 폴더
output_img_dir = os.path.join(path, "household_waste/dataset/images")
output_lbl_dir = os.path.join(path, "household_waste/dataset/labels")
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(output_img_dir, split), exist_ok=True)
    os.makedirs(os.path.join(output_lbl_dir, split), exist_ok=True)

# ===== 클래스 매핑 =====
class_map = {
    "비닐류": 0,
    "스티로폼류": 1,
    "종이류": 2,
    "캔류": 3,
    "페트병류": 4,
    "플라스틱류": 5
}

converted = 0

# ===== 1. JSON → YOLO TXT 변환 =====
for root, _, files in os.walk(json_dir):
    for filename in files:
        if not filename.lower().endswith(".json"):
            continue

        json_path = os.path.join(root, filename)
        try:
            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except json.JSONDecodeError:
            print(f"❌ JSON 에러: {filename}")
            continue

        bounding_list = data.get("Bounding", [])
        if not bounding_list:
            print(f"⚠️ 라벨 없음: {filename}")
            continue

        try:
            res_w, res_h = map(int, data.get("RESOLUTION", "1920*1080").split('*'))
        except:
            res_w, res_h = 1920, 1080  # 기본값

        yolo_lines = []
        for obj in bounding_list:
            try:
                drawing = obj.get("Drawing", "")
                if drawing == "POLYGON":
                    polygon_points = obj.get("PolygonPoint", [])
                    points = []
                    for pt in polygon_points:
                        coord_str = list(pt.values())[0]
                        x_str, y_str = coord_str.split(",")
                        points.append((float(x_str), float(y_str)))
                    xs = [p[0] for p in points]
                    ys = [p[1] for p in points]
                    x1, y1 = min(xs), min(ys)
                    x2, y2 = max(xs), max(ys)
                else:
                    x1 = float(obj["x1"])
                    y1 = float(obj["y1"])
                    x2 = float(obj["x2"])
                    y2 = float(obj["y2"])

                cx = (x1 + x2) / 2 / res_w
                cy = (y1 + y2) / 2 / res_h
                w = abs(x2 - x1) / res_w
                h = abs(y2 - y1) / res_h

                class_name = obj.get("CLASS", None)
                class_id = class_map.get(class_name, -1)
                if class_id == -1:
                    print(f"❓ 알 수 없는 클래스: {class_name} in {filename}")
                    continue

                yolo_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            except Exception as e:
                print(f"❌ 바운딩 오류: {e} in {filename}")
                continue

        if yolo_lines:
            output_name = os.path.splitext(filename)[0] + ".txt"
            output_path = os.path.join(yolo_label_dir, output_name)
            with open(output_path, "w") as out_f:
                out_f.write("\n".join(yolo_lines))
            converted += 1
        else:
            print(f"⚠️ 라벨 필터링 후 없음: {filename}")

print(f"\n🟢 YOLO 변환 완료: {converted} 개 변환됨")


# ===== 2. train/val/test 분할 =====
# 모든 JPG 이미지 가져오기
images = glob(os.path.join(image_dir, "**", "*.jpg"), recursive=True)
random.shuffle(images)

total = len(images)
train_split = int(total * 0.7)
val_split = int(total * 0.9)

train_imgs = images[:train_split]
val_imgs = images[train_split:val_split]
test_imgs = images[val_split:]

missing_labels = 0  # 라벨 없는 이미지 개수 카운트

def move_pairs(img_list, split):
    global missing_labels
    for img_path in img_list:
        fname = os.path.basename(img_path)
        label_path = os.path.join(yolo_label_dir, fname.replace(".jpg", ".txt"))

        if os.path.exists(label_path):
            shutil.copy(img_path, os.path.join(output_img_dir, split, fname))
            shutil.copy(label_path, os.path.join(output_lbl_dir, split, fname.replace(".jpg", ".txt")))
        else:
            missing_labels += 1
            print(f"❗ 라벨 없음 → {fname} 무시")

move_pairs(train_imgs, "train")
move_pairs(val_imgs, "val")
move_pairs(test_imgs, "test")

print(f"✅ 데이터 분리 완료: train={len(train_imgs)}, val={len(val_imgs)}, test={len(test_imgs)}")
print(f"⚠️ 라벨 없는 이미지 개수: {missing_labels} / 전체 {total}")


In [ ]:
# data.yaml 생성

import yaml
import os

# 저장할 경로
yaml_path = os.path.join("household_waste", "data.yaml")

# data.yaml 내용
data_config = {
    "train": "../dataset/images/train",
    "val": "../dataset/images/val",
    "test": "../dataset/images/test",
    "nc": 8,
    "names": {
        0: "비닐류",
        1: "스티로폼류",
        2: "종이류",
        3: "캔류",
        4: "페트병류",
        5: "플라스틱류",
        6: "담배꽁초",
        7: "기타"
    }
}

# yaml 파일 저장
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_config, f, allow_unicode=True)

print(f"✅ data.yaml 저장 완료: {yaml_path}")


✅ data.yaml 저장 완료: household_waste/data.yaml


In [ ]:
from ultralytics import YOLO
import os

# 이전에 학습한 모델 가중치 불러오기
model = YOLO(os.path.join(path, "trash_yolo/model_t/weights/best.pt"))

# 추가 학습
model.train(
    data=os.path.join(path, "household_waste/data.yaml"),  # 새 데이터셋
    epochs=30,         # 추가 학습할 epoch 수
    imgsz=640,         # 이미지 크기
    batch=16,          # 배치 사이즈
    project=os.path.join(path, "trash_yolo"),  # 저장 경로
    name="model_th",  # 결과 폴더 이름
    exist_ok=True
)


In [ ]:
# 한글 -> 영어로 바꿔서 출력

from ultralytics import YOLO
import os

model = YOLO(os.path.join(path, "trash_yolo/model_th/weights/best.pt"))

label_map = {
    "비닐류": "vinyl",
    "스티로폼류": "styrofoam",
    "종이류": "paper",
    "캔류": "can",
    "페트병류": "pet_bottle",
    "플라스틱류": "plastic",
    "담배꽁초": "cigarette",
    "기타": "etc"
}

results = model(os.path.join(path, "test_img/t10.jpg"))

# 한 번만 names를 영어로 치환
for result in results:
    result.names = {cls_id: label_map[name] for cls_id, name in result.names.items()}

# 이제 show()에도 영어가 나옴
results[0].show()


#vehicle_littering

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="LfukrBc5ekH621dAQE8Q")
project = rf.workspace("aivle777").project("vehicle_littering-jkmkn-2eyu3")
version = project.version(2)
dataset = version.download("yolov8")

In [ ]:
# train/val/test 분할

import os
import random
import shutil
from glob import glob

# 원본 데이터 경로
base_path = os.path.join(path, "vehicle_littering-2")

# 기존 train 폴더 내 이미지와 라벨 목록
all_images = glob(f"{base_path}/train/images/*.jpg")
random.shuffle(all_images)  # 데이터 섞기

# 전체 수 기준으로 비율 계산
total = len(all_images)
train_size = int(total * 0.7)
val_size = int(total * 0.2)
test_size = total - train_size - val_size  # 나머지

train_images = all_images[:train_size]
val_images = all_images[train_size:train_size + val_size]
test_images = all_images[train_size + val_size:]

# 폴더 생성
for split in ['train', 'valid', 'test']:
    os.makedirs(f"{base_path}/{split}/images", exist_ok=True)
    os.makedirs(f"{base_path}/{split}/labels", exist_ok=True)

# 이동 함수
def move_files(image_list, split):
    for img_path in image_list:
        label_path = img_path.replace('/images/', '/labels/').replace('.jpg', '.txt')
        shutil.move(img_path, img_path.replace('/train/', f'/{split}/'))
        shutil.move(label_path, label_path.replace('/train/', f'/{split}/'))

# 데이터 이동
move_files(train_images, 'train')
move_files(val_images, 'valid')
move_files(test_images, 'test')


In [ ]:
#필요한 클래스만 뽑아서 이미지 크롭

import os
import cv2
import shutil
import glob
from tqdm import tqdm

# 크롭할 클래스 지정 (필요하면 수정)
target_classes = ["box", "can", "plastic", "plastic_bag", "trash"]

# 클래스 이름 인덱스 매핑 (원본 dataset 기준)
class_name_to_id = {
    "box": 0,
    "can": 1,
    "car": 2,
    "car_door": 3,
    "hand": 4,
    "human": 5,
    "license_plate": 6,
    "plastic": 7,
    "plastic_bag": 8,
    "trash": 9,
}

# 선택한 클래스만 새로운 클래스 인덱스로 재매핑
remap_class_id = {
    0: 2,  # box
    1: 3,  # can
    7: 4,  # plastic
    8: 0,  # plastic_bag
    9: 7   # trash
}

# 크롭할 원본 클래스 id 목록
target_class_ids = list(remap_class_id.keys())

# base_path 설정
base_path = os.path.join(path, "vehicle_littering-2")
output_base = os.path.join(base_path, "cropped_dataset2")

# 기존 크롭 디렉토리 삭제
if os.path.exists(output_base):
    shutil.rmtree(output_base)

for split in ["train", "valid", "test"]:
    image_dir = os.path.join(base_path, split, "images")
    label_dir = os.path.join(base_path, split, "labels")

    output_img_dir = os.path.join(output_base, split, "images")
    output_label_dir = os.path.join(output_base, split, "labels")
    os.makedirs(output_img_dir, exist_ok=True)
    os.makedirs(output_label_dir, exist_ok=True)

    image_paths = glob.glob(os.path.join(image_dir, "*.jpg")) + glob.glob(os.path.join(image_dir, "*.png"))

    for image_path in tqdm(image_paths, desc=f"Processing {split}"):
        image_name = os.path.splitext(os.path.basename(image_path))[0]
        label_path = os.path.join(label_dir, image_name + ".txt")

        if not os.path.exists(label_path):
            continue

        image = cv2.imread(image_path)
        h, w = image.shape[:2]

        obj_idx = 0
        with open(label_path, "r") as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            # 최소 5개 이상이어야 처리 가능 (폴리곤도 가능)
            if len(parts) < 5:
                print(f"잘못된 라벨 형식: {line.strip()}")
                continue

            cls = int(parts[0])
            if cls not in target_class_ids:
                continue

            # YOLO bbox 형식 (5개) 인 경우
            if len(parts) == 5:
                x_center, y_center, bw, bh = map(float, parts[1:])
            else:
                # 폴리곤 좌표인 경우: (x1, y1, x2, y2, x3, y3, ...) → bbox 변환
                coords = list(map(float, parts[1:]))
                xs = coords[0::2]
                ys = coords[1::2]

                x_min = min(xs)
                x_max = max(xs)
                y_min = min(ys)
                y_max = max(ys)

                x_center = (x_min + x_max) / 2
                y_center = (y_min + y_max) / 2
                bw = x_max - x_min
                bh = y_max - y_min

            # 좌표 픽셀 단위 계산
            x1 = int((x_center - bw / 2) * w)
            y1 = int((y_center - bh / 2) * h)
            x2 = int((x_center + bw / 2) * w)
            y2 = int((y_center + bh / 2) * h)

            cropped_img = image[max(0, y1):min(y2, h), max(0, x1):min(x2, w)]
            if cropped_img.size == 0:
                continue

            new_filename = f"{image_name}_{obj_idx}"
            obj_idx += 1

            cv2.imwrite(os.path.join(output_img_dir, f"{new_filename}.jpg"), cropped_img)

            new_cls = remap_class_id[cls]
            with open(os.path.join(output_label_dir, f"{new_filename}.txt"), "w") as out_label:
                out_label.write(f"{new_cls} 0.5 0.5 1.0 1.0\n")

print("✅ 전체 train/val/test 크롭 완료!")


In [ ]:
from ultralytics import YOLO

model = YOLO(os.path.join(path, "trash_yolo/model_th/weights/best.pt"))

# 추가 학습
model.train(
    data=os.path.join(path, "vehicle_littering-2/data.yaml"),
    epochs=30,
    imgsz=640,
    batch=16,
    project=os.path.join(path, "trash_yolo"),
    name="model_thv",
    exist_ok=True
)


In [ ]:
model = YOLO(os.path.join(path, "trash_yolo/model_thv/weights/best.pt"))

results = model(os.path.join(path, "test_img/t2.jpg"))  # 예측
results[0].show()  # 이미지에 박스 그려진 결과 보기
